
This notebook trains **4 pre-trained CNN architectures** with fine-tuning on the HAM10000 dataset.

| Model | Type | Expected Accuracy |
|-------|------|-------------------|
| EfficientNetB3 | Pre-trained + Fine-tuned | ~75-80% |
| DenseNet121 | Pre-trained + Fine-tuned | ~72-76% |
| InceptionV3 | Pre-trained + Fine-tuned | ~69-72% |
| MobileNetV2 | Pre-trained + Fine-tuned | ~65-70% |

**Improvements over V1:**
- All 4 models are pre-trained (no weak from-scratch CNNs)
- Fine-tuning top layers for domain adaptation
- Data augmentation (rotation, flip, zoom, shift)
- 80/20 train/validation split for honest evaluation
- 3 epochs (still ~25 min on Apple M2 Air 8GB)
- Class weights to handle imbalanced dataset

In [ ]:
import pandas as pd
import numpy as np
import os
from glob import glob
from collections import Counter
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.applications import EfficientNetB3, DenseNet121, InceptionV3, MobileNetV2

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

print("TensorFlow:", tf.__version__)

In [ ]:
# --- Configuration ---
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 3
VALIDATION_SPLIT = 0.2
RANDOM_STATE = 42

CLASSES = [
    'akiec',  # Actinic keratoses
    'bcc',    # Basal cell carcinoma
    'bkl',    # Benign keratosis
    'df',     # Dermatofibroma
    'mel',    # Melanoma
    'nv',     # Melanocytic nevi
    'vasc',   # Vascular lesions
]

CLASS_LABELS = {
    'akiec': 'Actinic keratoses',
    'bcc': 'Basal cell carcinoma',
    'bkl': 'Benign keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic nevi',
    'vasc': 'Vascular lesions',
}

print(f"Config: {IMG_SIZE}, batch={BATCH_SIZE}, epochs={EPOCHS}")

In [ ]:
metadata = pd.read_csv('../data/HAM10000_metadata.csv')

image_paths = {
    os.path.splitext(os.path.basename(x))[0]: x
    for x in glob('../data/HAM10000_images_part_*/*.jpg')
}

metadata['path'] = metadata['image_id'].map(image_paths)
metadata = metadata.dropna()

metadata['label'] = metadata['dx']

In [ ]:
# --- Train/Validation Split ---
train_df, val_df = train_test_split(
    metadata, test_size=VALIDATION_SPLIT,
    random_state=RANDOM_STATE, stratify=metadata['dx']
)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# --- Compute Class Weights ---
class_weights = compute_class_weight(
    'balanced',
    classes=np.array(sorted(metadata['dx'].unique())),
    y=train_df['dx'].values
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f"\nClass weights: {class_weight_dict}")

In [ ]:
# --- Data Augmentation ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df, x_col='path', y_col='dx',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)

val_gen = val_datagen.flow_from_dataframe(
    dataframe=val_df, x_col='path', y_col='dx',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f"Class indices: {train_gen.class_indices}")

In [ ]:
# --- Model Architectures with Fine-Tuning Head ---

def build_classifier(base_model, name, unfreeze_top=20):
    """Attach a classification head and optionally fine-tune top N layers."""
    # Freeze all layers first
    for layer in base_model.layers:
        layer.trainable = False
    # Unfreeze top layers for fine-tuning
    for layer in base_model.layers[-unfreeze_top:]:
        layer.trainable = True
    
    x = GlobalAveragePooling2D()(base_model.output)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(7, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=out, name=name)
    return model

def build_efficientnet():
    base = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    return build_classifier(base, 'EfficientNetB3', unfreeze_top=30)

def build_densenet():
    base = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    return build_classifier(base, 'DenseNet121', unfreeze_top=25)

def build_inception():
    base = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    return build_classifier(base, 'InceptionV3', unfreeze_top=20)

def build_mobilenet():
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    return build_classifier(base, 'MobileNetV2', unfreeze_top=20)

print("✅ Model builders ready")

In [ ]:
# --- 🏆 Tournament V2 ---
competitors = {
    'EfficientNetB3': build_efficientnet,
    'DenseNet121': build_densenet,
    'InceptionV3': build_inception,
    'MobileNetV2': build_mobilenet,
}

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, verbose=1),
]

results = []
best_val_acc = 0.0
winning_model_name = ''

print('🏟️  Tournament V2 — 4 Pre-trained Models with Fine-Tuning')
print(f'⏱️  Config: {EPOCHS} epochs, batch={BATCH_SIZE}, val_split={VALIDATION_SPLIT}')
print('=' * 60)

for name, builder in competitors.items():
    print(f'\n🔄 Training: {name}...')
    tf.keras.backend.clear_session()
    
    model = builder()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    trainable_count = sum(1 for l in model.layers if l.trainable)
    total_params = model.count_params()
    print(f'   Params: {total_params:,} | Trainable layers: {trainable_count}')
    
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )
    
    val_acc = max(history.history['val_accuracy'])
    train_acc = max(history.history['accuracy'])
    val_loss = min(history.history['val_loss'])
    
    result = {
        'name': name,
        'val_accuracy': round(val_acc * 100, 2),
        'train_accuracy': round(train_acc * 100, 2),
        'val_loss': round(val_loss, 4),
        'params': f'{total_params/1e6:.1f}M',
        'trainable_layers': trainable_count,
    }
    results.append(result)
    
    print(f'   ✅ Val Accuracy: {val_acc*100:.2f}% | Train Accuracy: {train_acc*100:.2f}%')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        winning_model_name = name
        model.save('../backend/skin_disease_model.h5')
        print(f'   💾 New best! Saved to ../backend/skin_disease_model.h5')

print('\n' + '=' * 60)
print(f'🏆 CHAMPION: {winning_model_name}')
print(f'📊 Validation Accuracy: {best_val_acc*100:.2f}%')
print('=' * 60)

In [ ]:
# --- Results Summary ---
results_df = pd.DataFrame(results).sort_values('val_accuracy', ascending=False)
results_df['rank'] = range(1, len(results_df) + 1)
results_df['winner'] = results_df['name'] == winning_model_name

print('\n📊 Tournament Leaderboard:')
print(results_df[['rank', 'name', 'val_accuracy', 'train_accuracy', 'val_loss', 'params']].to_string(index=False))

# Save results for frontend reference
results_json = results_df.to_dict('records')
with open('../backend/tournament_results.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print('\n✅ Results saved to ../backend/tournament_results.json')